### Leitura do arquivo de treinamento

O arquivo train.csv contém os dados que serão usados para desenvolver
os modelos de predição. 

Primeiro, lemos o cabeçalho do treinamento para identificar os nomes
das colunas.

Usamos o módulo csv para ler o arquivo e Path para indicar sua
localização. 


In [1]:
from pathlib import Path
import csv

caminho = Path("kaggle-training/train.csv")

with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    leitor = csv.reader(arquivo)
    colunas = next(leitor)

print(colunas)

['id', 'PATIENT_CODE', 'EXAM_DATE', 'COLLECTION_SITE', 'LAB_DEVICE', 'STUDY_PROTOCOL', 'GENDER', 'AGE', 'BIRTH_YEAR', 'HEIGHT', 'WEIGHT', 'BLOOD_PRESSURE', 'BLOOD_TYPE', 'SMOKER', 'PHYSICAL_ACTIVITY', 'FAMILY_HISTORY', 'GLUCOSE', 'HEMOGLOBIN', 'TRIGLYCERIDES', 'SODIUM', 'class']


### Quantidade de amostras e distribuição das classes

Lemos as linhas do treinamento e contamos quantas amostras pertencem
a cada classe. Essa contagem permite verificar se há diferença na
quantidade de exemplos de controle e diabetes.

Usamos DictReader para acessar os valores pelo nome de cada coluna
e Counter para contar as ocorrências dos rótulos.

In [2]:
from collections import Counter

with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
    amostras = list(csv.DictReader(arquivo))

contagem_classes = Counter(amostra["class"] for amostra in amostras)

print("Total de amostras:", len(amostras))
print("Amostras por classe:", dict(contagem_classes))

Total de amostras: 540
Amostras por classe: {'1': 281, '0': 259}


### Identificação, idade, altura e peso

Verificamos se há valores ausentes em PATIENT_CODE, GENDER, AGE,
HEIGHT e WEIGHT. Também conferimos exemplos dos valores para planejar
o tratamento de cada coluna.

In [3]:
colunas_analisar = ["PATIENT_CODE", "GENDER", "AGE", "HEIGHT", "WEIGHT"]

for coluna in colunas_analisar:
    valores = [amostra[coluna].strip() for amostra in amostras]
    ausentes = sum(1 for valor in valores if valor == "")

    print(f"{coluna}:")
    print(f"  Valores vazios: {ausentes}")

    if coluna != "PATIENT_CODE":
        exemplos = list(dict.fromkeys(valores))[:5]
        print(f"  Exemplos de valores: {exemplos}")

    print()

PATIENT_CODE:
  Valores vazios: 0

GENDER:
  Valores vazios: 6
  Exemplos de valores: ['MALE', 'f', 'FEMALE', 'F', 'female']

AGE:
  Valores vazios: 5
  Exemplos de valores: ['34', '59', '44', '49', '58']

HEIGHT:
  Valores vazios: 34
  Exemplos de valores: ['170', '161', '159', '1.66', '172']

WEIGHT:
  Valores vazios: 13
  Exemplos de valores: ['83.5', '76.6', '59.3', '46.1', '88.2']



#### Valores de gênero

Contamos as ocorrências de cada valor registrado em GENDER para
identificar grafias diferentes e planejar a padronização.

In [4]:
generos = [amostra["GENDER"].strip() for amostra in amostras]

contagem_generos = Counter(generos)

for genero, quantidade in contagem_generos.items():
    print(f"{genero!r}: {quantidade}")

'MALE': 181
'f': 9
'FEMALE': 307
'F': 11
'female': 5
'm': 4
'male': 5
'': 6
'Female': 7
'M': 3
'Male': 2


#### Padronização de gênero

Padronizamos as diferentes grafias de gênero em três categorias:
M para masculino, F para feminino e N para não informado.

Criamos uma cópia das amostras para preservar os valores originais
durante a análise.

In [5]:
amostras_padronizadas = [amostra.copy() for amostra in amostras]

mapa_genero = {
    "MALE": "M",
    "M": "M",
    "FEMALE": "F",
    "F": "F",
    "": "N"
}

for amostra in amostras_padronizadas:
    genero_original = amostra["GENDER"].strip().upper()
    amostra["GENDER"] = mapa_genero[genero_original]

contagem_padronizada = Counter(
    amostra["GENDER"] for amostra in amostras_padronizadas
)

print(dict(contagem_padronizada))

{'M': 195, 'F': 339, 'N': 6}


#### Verificação das alturas

Antes de padronizar a altura em centímetros, verificamos quantos
valores são numéricos, quantos estão vazios e se há valores que
não conseguimos converter para número.

Também observamos o menor e o maior valor numérico registrado.

In [6]:
alturas_numericas = []
alturas_invalidas = []
alturas_vazias = 0

for amostra in amostras_padronizadas:
    altura = amostra["HEIGHT"].strip().replace(",", ".")

    if altura == "":
        alturas_vazias += 1
    else:
        try:
            alturas_numericas.append(float(altura))
        except ValueError:
            alturas_invalidas.append(altura)

print("Alturas vazias:", alturas_vazias)
print("Alturas numéricas:", len(alturas_numericas))
print("Valores não numéricos:", sorted(set(alturas_invalidas)))

if alturas_numericas:
    print("Menor altura registrada:", min(alturas_numericas))
    print("Maior altura registrada:", max(alturas_numericas))

Alturas vazias: 34
Alturas numéricas: 506
Valores não numéricos: []
Menor altura registrada: 1.47
Maior altura registrada: 187.0


#### Formatos de altura

Separamos as alturas numéricas em três grupos: valores menores que 3,
valores a partir de 100 e valores entre esses limites.

Essa verificação ajuda a identificar registros que precisam ser
examinados antes da conversão para centímetros.

In [7]:
alturas_em_metros = [altura for altura in alturas_numericas if altura < 3]

alturas_em_centimetros = [
    altura for altura in alturas_numericas if altura >= 100
]

alturas_a_verificar = [
    altura for altura in alturas_numericas if 3 <= altura < 100
]

print("Possivelmente em metros:", len(alturas_em_metros))
print("Possivelmente em centímetros:", len(alturas_em_centimetros))
print("Valores a verificar:", alturas_a_verificar)

Possivelmente em metros: 107
Possivelmente em centímetros: 399
Valores a verificar: []


#### Padronização da altura

Padronizamos as alturas em centímetros. Valores menores que 3 são
interpretados como metros e multiplicados por 100; os demais permanecem
na escala em que foram registrados.

Os campos vazios continuam sem valor nesta etapa.

In [8]:
for amostra in amostras_padronizadas:
    altura_texto = str(amostra["HEIGHT"]).strip().replace(",", ".")

    if altura_texto != "":
        altura = float(altura_texto)

        if altura < 3:
            altura = altura * 100

        amostra["HEIGHT"] = altura

#### Conferência das alturas padronizadas

Conferimos a quantidade de alturas preenchidas e seus valores mínimo
e máximo após a conversão para centímetros.

In [9]:
alturas_cm = [
    amostra["HEIGHT"]
    for amostra in amostras_padronizadas
    if amostra["HEIGHT"] != ""
]

print("Alturas preenchidas:", len(alturas_cm))
print("Menor altura em cm:", min(alturas_cm))
print("Maior altura em cm:", max(alturas_cm))

Alturas preenchidas: 506
Menor altura em cm: 143.0
Maior altura em cm: 187.0


Após a padronização, as 506 alturas preenchidas variam de 143 cm
a 187 cm. Os 34 campos vazios permanecem sem preenchimento.

#### Verificação dos pesos

Verificamos quantos pesos estão preenchidos e se os valores podem ser
convertidos para números. Também observamos o menor e o maior peso
registrado antes de decidir como tratar a coluna.

In [10]:
pesos_numericos = []
pesos_nao_convertidos = []
pesos_vazios = 0

for amostra in amostras_padronizadas:
    peso = amostra["WEIGHT"].strip().replace(",", ".")

    if peso == "":
        pesos_vazios += 1
    else:
        try:
            pesos_numericos.append(float(peso))
        except ValueError:
            pesos_nao_convertidos.append(peso)

print("Pesos vazios:", pesos_vazios)
print("Pesos numéricos:", len(pesos_numericos))
print("Valores não convertidos:", sorted(set(pesos_nao_convertidos)))

if pesos_numericos:
    print("Menor peso registrado:", min(pesos_numericos))
    print("Maior peso registrado:", max(pesos_numericos))

Pesos vazios: 13
Pesos numéricos: 527
Valores não convertidos: []
Menor peso registrado: 41.3
Maior peso registrado: 242.1


#### Análise dos pesos mais altos

Observamos os pesos acima de 150 para verificar quantos registros
aparecem nessa faixa. O limite é usado apenas para investigação;
um valor alto não é necessariamente um erro.

In [11]:
pesos_altos = [peso for peso in pesos_numericos if peso > 150]

print("Quantidade de pesos acima de 150:", len(pesos_altos))
print("Pesos encontrados:", sorted(pesos_altos))

Quantidade de pesos acima de 150: 21
Pesos encontrados: [153.7, 158.1, 164.5, 168.9, 175.9, 179.0, 179.0, 179.2, 184.5, 186.3, 186.3, 187.2, 187.2, 192.7, 194.7, 211.9, 212.5, 222.7, 227.3, 234.3, 242.1]


#### Peso em relação à idade e à altura

Examinamos as amostras com os cinco menores e os cinco maiores pesos,
junto com a idade e a altura de cada uma.

Essa análise ajuda a investigar os valores, mas não confirma a unidade
do peso. A informação sobre a unidade permanece pendente.

In [12]:
amostras_com_peso = []

for amostra in amostras_padronizadas:
    peso_texto = amostra["WEIGHT"].strip().replace(",", ".")

    if peso_texto != "":
        peso = float(peso_texto)
        amostras_com_peso.append((peso, amostra))

amostras_ordenadas = sorted(amostras_com_peso, key=lambda item: item[0])

print("Cinco menores pesos:")
for peso, amostra in amostras_ordenadas[:5]:
    print("Peso:", peso, "| Idade:", amostra["AGE"], "| Altura em cm:", amostra["HEIGHT"])

print("\nCinco maiores pesos:")
for peso, amostra in amostras_ordenadas[-5:]:
    print("Peso:", peso, "| Idade:", amostra["AGE"], "| Altura em cm:", amostra["HEIGHT"])

Cinco menores pesos:
Peso: 41.3 | Idade: 44 | Altura em cm: 146.0
Peso: 41.4 | Idade: 44 | Altura em cm: 154.0
Peso: 41.5 | Idade: 58 | Altura em cm: 153.0
Peso: 42.4 | Idade: 72 | Altura em cm: 151.0
Peso: 42.6 | Idade: 76 | Altura em cm: 154.0

Cinco maiores pesos:
Peso: 212.5 | Idade: 41 | Altura em cm: 179.0
Peso: 222.7 | Idade: 60 | Altura em cm: 
Peso: 227.3 | Idade: 58 | Altura em cm: 172.0
Peso: 234.3 | Idade: 54 | Altura em cm: 174.0
Peso: 242.1 | Idade: 53 | Altura em cm: 170.0


### Distribuição dos pesos

Calculamos a mediana e os quartis para entender como os pesos estão
distribuídos. Também identificamos valores afastados pelo critério
do intervalo interquartil.

Um valor identificado como outlier não é necessariamente um erro.

In [13]:
from statistics import median, quantiles

mediana_pesos = median(pesos_numericos)
q1, q2, q3 = quantiles(pesos_numericos, n=4, method="inclusive")

intervalo_interquartil = q3 - q1
limite_inferior = q1 - 1.5 * intervalo_interquartil
limite_superior = q3 + 1.5 * intervalo_interquartil

pesos_fora_limites = [
    peso for peso in pesos_numericos
    if peso < limite_inferior or peso > limite_superior
]

print("Mediana:", mediana_pesos)
print("Primeiro quartil (Q1):", q1)
print("Terceiro quartil (Q3):", q3)
print("Intervalo interquartil:", intervalo_interquartil)
print("Limites:", limite_inferior, "a", limite_superior)
print("Quantidade fora dos limites:", len(pesos_fora_limites))
print("Pesos fora dos limites:", sorted(pesos_fora_limites))

Mediana: 76.0
Primeiro quartil (Q1): 63.9
Terceiro quartil (Q3): 87.25
Intervalo interquartil: 23.35
Limites: 28.874999999999993 a 122.275
Quantidade fora dos limites: 32
Pesos fora dos limites: [125.2, 125.2, 130.2, 130.5, 131.2, 134.7, 138.0, 141.5, 149.0, 149.5, 149.5, 153.7, 158.1, 164.5, 168.9, 175.9, 179.0, 179.0, 179.2, 184.5, 186.3, 186.3, 187.2, 187.2, 192.7, 194.7, 211.9, 212.5, 222.7, 227.3, 234.3, 242.1]


Adotamos provisoriamente a premissa de que WEIGHT está em quilogramas.
A distribuição dos valores apoia essa hipótese, mas não confirma a
unidade de cada registro.

Os valores extremos foram mantidos. A unidade será confirmada com
o professor posteriormente.

#### Padronização numérica do peso

Convertemos os valores preenchidos de WEIGHT para números decimais e mantemos os campos vazios. Preservamos os valores extremos e a premissa provisória de que os pesos estão em quilogramas, sem conversão de unidade.


In [14]:
for amostra in amostras_padronizadas:
    peso_texto = str(amostra["WEIGHT"]).strip().replace(",", ".")

    if peso_texto != "":
        amostra["WEIGHT"] = float(peso_texto)

pesos_preenchidos = [
    amostra["WEIGHT"]
    for amostra in amostras_padronizadas
    if amostra["WEIGHT"] != ""
]

print("Pesos numéricos:", len(pesos_preenchidos))
print("Campos vazios:", len(amostras_padronizadas) - len(pesos_preenchidos))
print("Menor peso:", min(pesos_preenchidos))
print("Maior peso:", max(pesos_preenchidos))


Pesos numéricos: 527
Campos vazios: 13
Menor peso: 41.3
Maior peso: 242.1


#### Verificação das idades

Verificamos os valores vazios e os formatos registrados em AGE.
Também observamos a menor e a maior idade para identificar valores
que precisam ser investigados.

In [15]:
idades_numericas = []
idades_nao_convertidas = []
idades_vazias = 0

for amostra in amostras_padronizadas:
    idade = amostra["AGE"].strip()

    if idade == "":
        idades_vazias += 1
    else:
        try:
            idades_numericas.append(int(idade))
        except ValueError:
            idades_nao_convertidas.append(idade)

print("Idades vazias:", idades_vazias)
print("Idades numéricas inteiras:", len(idades_numericas))
print("Valores não convertidos:", sorted(set(idades_nao_convertidas)))

if idades_numericas:
    print("Menor idade:", min(idades_numericas))
    print("Maior idade:", max(idades_numericas))

Idades vazias: 5
Idades numéricas inteiras: 535
Valores não convertidos: []
Menor idade: -1
Maior idade: 999


#### Investigação de idades incomuns

Contamos as idades negativas ou 0 e as maiores que 100 para identificar
valores que precisam ser investigados. Esses limites são usados
apenas para análise; nenhum valor é alterado nesta etapa.

In [16]:
idades_incomuns = [
    idade for idade in idades_numericas
    if idade < 1 or idade > 100
]

contagem_incomuns = Counter(idades_incomuns)

print("Quantidade de idades incomuns:", len(idades_incomuns))
print("Valores encontrados:", dict(contagem_incomuns))

Quantidade de idades incomuns: 6
Valores encontrados: {0: 2, 999: 3, -1: 1}


#### Distribuição das idades

Observamos a menor idade, o primeiro quartil, a mediana, o terceiro
quartil e a maior idade entre os valores de 1 a 100 anos.

Os campos vazios e os valores fora desse intervalo não entram nesta
análise. O limite é provisório e usado para investigação.

In [17]:
idades_para_analise = [
    idade for idade in idades_numericas
    if 1 <= idade <= 100
]

q1_idade, mediana_idade, q3_idade = quantiles(
    idades_para_analise,
    n=4,
    method="inclusive"
)

print("Quantidade analisada:", len(idades_para_analise))
print("Menor idade:", min(idades_para_analise))
print("Primeiro quartil:", q1_idade)
print("Mediana:", mediana_idade)
print("Terceiro quartil:", q3_idade)
print("Maior idade:", max(idades_para_analise))

Quantidade analisada: 529
Menor idade: 16
Primeiro quartil: 49.0
Mediana: 57.0
Terceiro quartil: 67.0
Maior idade: 98


#### Investigação das idades ausentes ou incomuns

Observamos o ano de nascimento e a data do exame das amostras com
idade vazia, zero, negativa ou maior que 100.

Essas informações podem ajudar a identificar inconsistências, mas
não alteramos as idades nesta etapa.

In [18]:
idades_investigar = ["", "0", "-1", "999"]

for amostra in amostras_padronizadas:
    idade = amostra["AGE"].strip()

    if idade in idades_investigar:
        print(
            "Idade:", repr(idade),
            "| Ano de nascimento:", amostra["BIRTH_YEAR"],
            "| Data do exame:", amostra["EXAM_DATE"]
        )

Idade: '' | Ano de nascimento: 1976 | Data do exame: 13/07/2021
Idade: '' | Ano de nascimento: 1947 | Data do exame: 2019-03-08
Idade: '0' | Ano de nascimento: 1992 | Data do exame: 2021-10-19
Idade: '999' | Ano de nascimento: 1959 | Data do exame: Nov 27 2018
Idade: '-1' | Ano de nascimento: 1966 | Data do exame: 2020-03-05
Idade: '0' | Ano de nascimento: 1980 | Data do exame: 2018-08-28
Idade: '' | Ano de nascimento: 1944 | Data do exame: 2019-06-16
Idade: '' | Ano de nascimento: 1977 | Data do exame: 2019-02-24
Idade: '' | Ano de nascimento: 1970 | Data do exame: 2019-04-14
Idade: '999' | Ano de nascimento: 1962 | Data do exame: Jul 8 2021
Idade: '999' | Ano de nascimento: 1954 | Data do exame: 30/06/2018


#### Comparação da idade com o ano de nascimento

Comparamos AGE com BIRTH_YEAR e o ano de EXAM_DATE no treinamento.

Como não temos o dia e o mês de nascimento, consideramos compatíveis
duas idades possíveis: a diferença entre os anos e essa diferença
menos um.

Os registros ausentes ou que não puderem ser interpretados são
contados separadamente. Nenhum valor é alterado.

In [19]:
from datetime import datetime

formatos_data = ["%Y-%m-%d", "%d/%m/%Y", "%b %d %Y"]

compativeis = 0
divergentes = 0
nao_comparados = 0

for amostra in amostras_padronizadas:
    idade_texto = amostra["AGE"].strip()
    nascimento_texto = amostra["BIRTH_YEAR"].strip()
    data_texto = amostra["EXAM_DATE"].strip()

    ano_exame = None

    for formato in formatos_data:
        try:
            ano_exame = datetime.strptime(data_texto, formato).year
            break
        except ValueError:
            continue

    try:
        idade = int(idade_texto)
        ano_nascimento = int(nascimento_texto)
    except ValueError:
        nao_comparados += 1
        continue

    if ano_exame is None or not 1 <= idade <= 100:
        nao_comparados += 1
        continue

    diferenca_anos = ano_exame - ano_nascimento

    if idade in (diferenca_anos - 1, diferenca_anos):
        compativeis += 1
    else:
        divergentes += 1

print("Idades compatíveis:", compativeis)
print("Idades divergentes:", divergentes)
print("Não comparadas:", nao_comparados)

Idades compatíveis: 519
Idades divergentes: 0
Não comparadas: 21


#### Idades possíveis e registros não comparados

Calculamos as duas idades possíveis para os registros com idade
vazia ou incomum, usando o ano de nascimento e o ano do exame.

Também investigamos os demais registros não comparados para identificar
quais informações faltam ou não puderam ser interpretadas.

Os valores originais são preservados.

In [20]:
idades_investigar = {"", "0", "-1", "999"}

print("Idades vazias ou incomuns:")

for amostra in amostras_padronizadas:
    idade_texto = amostra["AGE"].strip()

    if idade_texto in idades_investigar:
        nascimento_texto = amostra["BIRTH_YEAR"].strip()
        data_texto = amostra["EXAM_DATE"].strip()

        ano_exame = None

        for formato in formatos_data:
            try:
                ano_exame = datetime.strptime(data_texto, formato).year
                break
            except ValueError:
                continue

        try:
            ano_nascimento = int(nascimento_texto)
        except ValueError:
            ano_nascimento = None

        if ano_exame is not None and ano_nascimento is not None:
            diferenca = ano_exame - ano_nascimento
            print(
                "Idade registrada:", repr(idade_texto),
                "| Idades possíveis:", diferenca - 1, "ou", diferenca
            )
        else:
            print(
                "Idade registrada:", repr(idade_texto),
                "| Não foi possível calcular as idades"
            )

print("\nOutros registros não comparados:")

for amostra in amostras_padronizadas:
    idade_texto = amostra["AGE"].strip()

    if idade_texto in idades_investigar:
        continue

    nascimento_texto = amostra["BIRTH_YEAR"].strip()
    data_texto = amostra["EXAM_DATE"].strip()
    motivos = []

    try:
        idade = int(idade_texto)
        if not 1 <= idade <= 100:
            motivos.append("idade fora do intervalo de análise")
    except ValueError:
        motivos.append("idade não convertida")

    try:
        int(nascimento_texto)
    except ValueError:
        motivos.append("ano de nascimento vazio ou inválido")

    data_reconhecida = False

    for formato in formatos_data:
        try:
            datetime.strptime(data_texto, formato)
            data_reconhecida = True
            break
        except ValueError:
            continue

    if not data_reconhecida:
        motivos.append("data do exame não reconhecida")

    if motivos:
        print(
            "Idade registrada:", repr(idade_texto),
            "| Motivos:", ", ".join(motivos)
        )

Idades vazias ou incomuns:
Idade registrada: '' | Idades possíveis: 44 ou 45
Idade registrada: '' | Idades possíveis: 71 ou 72
Idade registrada: '0' | Idades possíveis: 28 ou 29
Idade registrada: '999' | Idades possíveis: 58 ou 59
Idade registrada: '-1' | Idades possíveis: 53 ou 54
Idade registrada: '0' | Idades possíveis: 37 ou 38
Idade registrada: '' | Idades possíveis: 74 ou 75
Idade registrada: '' | Idades possíveis: 41 ou 42
Idade registrada: '' | Idades possíveis: 48 ou 49
Idade registrada: '999' | Idades possíveis: 58 ou 59
Idade registrada: '999' | Idades possíveis: 63 ou 64

Outros registros não comparados:
Idade registrada: '80' | Motivos: ano de nascimento vazio ou inválido
Idade registrada: '56' | Motivos: ano de nascimento vazio ou inválido
Idade registrada: '56' | Motivos: ano de nascimento vazio ou inválido
Idade registrada: '59' | Motivos: ano de nascimento vazio ou inválido
Idade registrada: '56' | Motivos: ano de nascimento vazio ou inválido
Idade registrada: '65' | M

#### Resumo da análise de idade

A coluna AGE possui 540 registros. Encontramos 5 campos vazios e
6 valores incomuns: dois valores 0, um valor -1 e três valores 999.

Para os 11 registros com idade vazia ou incomum, calculamos duas
idades possíveis a partir do ano de nascimento e do ano do exame.
Como não temos o dia e o mês de nascimento, não é possível determinar
a idade exata.

Entre os demais registros, 519 idades foram compatíveis com o ano
de nascimento e a data do exame. Outros 10 não puderam ser comparados
porque o ano de nascimento estava vazio ou não pôde ser interpretado.

Na análise das 529 idades entre 1 e 100 anos, a mediana foi 57 anos
e a metade central ficou entre 49 e 67 anos.

As idades originais foram mantidas. A escolha entre usar a idade
numérica ou faixas etárias será avaliada nos experimentos. Também
verificaremos se as duas idades possíveis dos registros investigados
ficam na mesma faixa escolhida.

### Amostras por paciente

Contamos quantas amostras estão associadas a cada PATIENT_CODE.
Essa análise ajuda a identificar pacientes com mais de um registro
e será importante na divisão entre treinamento e validação.

In [21]:
contagem_pacientes = Counter(
    amostra["PATIENT_CODE"].strip()
    for amostra in amostras_padronizadas
)

quantidade_por_paciente = Counter(contagem_pacientes.values())

print("Quantidade de pacientes:", len(contagem_pacientes))

for quantidade, pacientes in sorted(quantidade_por_paciente.items()):
    print(f"Pacientes com {quantidade} amostra(s): {pacientes}")

Quantidade de pacientes: 520
Pacientes com 1 amostra(s): 500
Pacientes com 2 amostra(s): 20


O treinamento contém 520 códigos de paciente: 500 aparecem em uma
amostra e 20 aparecem em duas, totalizando 540 amostras.

Na divisão entre treinamento e validação, as amostras de um mesmo
paciente deverão permanecer no mesmo grupo.

#### Datas dos pacientes com duas amostras

Comparamos as datas de exame dos pacientes que possuem duas amostras.
Antes da comparação, interpretamos as datas nos formatos encontrados
na base.

Contamos os pacientes com datas iguais, datas diferentes e datas que
não puderam ser interpretadas.

In [22]:
amostras_por_paciente = {}

for amostra in amostras_padronizadas:
    codigo = amostra["PATIENT_CODE"].strip()

    if codigo not in amostras_por_paciente:
        amostras_por_paciente[codigo] = []

    amostras_por_paciente[codigo].append(amostra)

datas_iguais = 0
datas_diferentes = 0
datas_nao_comparadas = 0

for amostras_paciente in amostras_por_paciente.values():
    if len(amostras_paciente) == 2:
        datas = []

        for amostra in amostras_paciente:
            data_texto = amostra["EXAM_DATE"].strip()
            data_convertida = None

            for formato in formatos_data:
                try:
                    data_convertida = datetime.strptime(data_texto, formato).date()
                    break
                except ValueError:
                    continue

            datas.append(data_convertida)

        if None in datas:
            datas_nao_comparadas += 1
        elif datas[0] == datas[1]:
            datas_iguais += 1
        else:
            datas_diferentes += 1

print("Pacientes com datas iguais:", datas_iguais)
print("Pacientes com datas diferentes:", datas_diferentes)
print("Pacientes não comparados:", datas_nao_comparadas)

Pacientes com datas iguais: 20
Pacientes com datas diferentes: 0
Pacientes não comparados: 0


#### Comparação das amostras do mesmo paciente

Comparamos as duas amostras dos pacientes que aparecem duas vezes.
A coluna id não entra na comparação, pois identifica cada amostra.

Contamos os pares com valores iguais e, para os pares diferentes,
registramos em quais colunas há diferenças.

In [23]:
pares_iguais = 0
pares_diferentes = 0
diferencas_por_coluna = Counter()

for amostras_paciente in amostras_por_paciente.values():
    if len(amostras_paciente) == 2:
        primeira = amostras_paciente[0]
        segunda = amostras_paciente[1]

        colunas_diferentes = []

        for coluna in primeira:
            if coluna != "id" and primeira[coluna] != segunda[coluna]:
                colunas_diferentes.append(coluna)
                diferencas_por_coluna[coluna] += 1

        if colunas_diferentes:
            pares_diferentes += 1
        else:
            pares_iguais += 1

print("Pares iguais, exceto pelo id:", pares_iguais)
print("Pares com alguma diferença:", pares_diferentes)
print("Diferenças por coluna:", dict(diferencas_por_coluna))

Pares iguais, exceto pelo id: 20
Pares com alguma diferença: 0
Diferenças por coluna: {}


#### Remoção de amostras repetidas

Após a padronização, os 20 pares de amostras do mesmo paciente são
iguais em todas as colunas, exceto id.

Mantemos a primeira amostra encontrada para cada PATIENT_CODE. A lista
original de amostras padronizadas é preservada para consulta.

In [24]:
pacientes_unicos = {}

for amostra in amostras_padronizadas:
    codigo = amostra["PATIENT_CODE"].strip()

    if codigo not in pacientes_unicos:
        pacientes_unicos[codigo] = amostra.copy()

amostras_sem_repeticao = list(pacientes_unicos.values())

print("Amostras antes:", len(amostras_padronizadas))
print("Amostras depois:", len(amostras_sem_repeticao))
print("Amostras retiradas:", len(amostras_padronizadas) - len(amostras_sem_repeticao))

Amostras antes: 540
Amostras depois: 520
Amostras retiradas: 20


#### Resumo das amostras repetidas

O treinamento original contém 540 amostras e 520 códigos de paciente.
Desses pacientes, 500 possuem uma amostra e 20 possuem duas.

As duas amostras de cada um dos 20 pacientes têm a mesma data de exame.
Após as padronizações de gênero e altura, cada par também apresenta
valores iguais nas demais colunas, exceto pelo id.

Mantivemos a primeira amostra encontrada de cada paciente, formando
uma lista com 520 amostras. A lista anterior, com 540 amostras, e o
arquivo original foram preservados.

### Formatos das datas de exame

Contamos os formatos encontrados em EXAM_DATE na base sem repetição.
Essa verificação ajuda a identificar datas vazias ou que não seguem
os formatos conhecidos.

In [25]:
contagem_formatos = Counter()
datas_nao_reconhecidas = []

for amostra in amostras_sem_repeticao:
    data_texto = amostra["EXAM_DATE"].strip()
    formato_encontrado = None

    for formato in formatos_data:
        try:
            datetime.strptime(data_texto, formato)
            formato_encontrado = formato
            break
        except ValueError:
            continue

    if formato_encontrado is None:
        datas_nao_reconhecidas.append(data_texto)
    else:
        contagem_formatos[formato_encontrado] += 1

print("Datas por formato:", dict(contagem_formatos))
print("Datas não reconhecidas:", len(datas_nao_reconhecidas))

Datas por formato: {'%Y-%m-%d': 300, '%b %d %Y': 59, '%d/%m/%Y': 161}
Datas não reconhecidas: 0


#### Padronização das datas de exame

Convertamos as datas de EXAM_DATE para o formato AAAA-MM-DD.
A conversão é aplicada à lista sem repetição; o arquivo original
e a lista anterior permanecem preservados.

In [26]:
for amostra in amostras_sem_repeticao:
    data_texto = amostra["EXAM_DATE"].strip()

    for formato in formatos_data:
        try:
            data_convertida = datetime.strptime(data_texto, formato)
            amostra["EXAM_DATE"] = data_convertida.strftime("%Y-%m-%d")
            break
        except ValueError:
            continue
    else:
        raise ValueError(f"Data não reconhecida: {data_texto!r}")

print("Datas padronizadas:", len(amostras_sem_repeticao))

Datas padronizadas: 520


#### Conferência das datas padronizadas

Verificamos se todas as datas da lista sem repetição seguem o formato
AAAA-MM-DD após a conversão.

In [27]:
datas_fora_padrao = []

for amostra in amostras_sem_repeticao:
    data_texto = amostra["EXAM_DATE"]

    try:
        data = datetime.strptime(data_texto, "%Y-%m-%d")

        if data.strftime("%Y-%m-%d") != data_texto:
            datas_fora_padrao.append(data_texto)
    except ValueError:
        datas_fora_padrao.append(data_texto)

print("Datas fora do padrão:", len(datas_fora_padrao))
print("Valores encontrados:", datas_fora_padrao)

Datas fora do padrão: 0
Valores encontrados: []


#### Resumo das datas de exame

Na base sem repetição, encontramos datas em três formatos diferentes.
As 520 datas foram interpretadas e padronizadas para AAAA-MM-DD.

Após a conversão, nenhuma data ficou fora do formato escolhido.

### Verificação da pressão arterial

Verificamos os valores vazios e observamos exemplos de como a pressão
arterial está registrada na base sem repetição.

Essa análise ajuda a identificar o formato dos dados antes de definir
qualquer padronização.

In [28]:
pressoes = [
    amostra["BLOOD_PRESSURE"].strip()
    for amostra in amostras_sem_repeticao
]

pressoes_vazias = pressoes.count("")
exemplos_pressao = list(dict.fromkeys(pressoes))[:15]

print("Campos vazios:", pressoes_vazias)
print("Exemplos de valores:", exemplos_pressao)

Campos vazios: 22
Exemplos de valores: ['113/87', '114 / 75', '?', '95/63', '115/83', '104/77', '138/77', '134/96', '136/80', '140/87', '120/86', '99/64', '117/81', '98/72', '122/82']


#### Formatos da pressão arterial

Classificamos os valores de BLOOD_PRESSURE em campos vazios, símbolo
de interrogação, pares de números separados por barra e outros formatos.

Essa verificação não altera os dados.

In [29]:
formatos_pressao = Counter()
outros_formatos = []

for pressao in pressoes:
    if pressao == "":
        formatos_pressao["vazio"] += 1
    elif pressao == "?":
        formatos_pressao["interrogação"] += 1
    else:
        partes = pressao.split("/")

        if (
            len(partes) == 2
            and partes[0].strip().isdigit()
            and partes[1].strip().isdigit()
        ):
            formatos_pressao["dois números"] += 1
        else:
            formatos_pressao["outros"] += 1
            outros_formatos.append(pressao)

print("Quantidade por formato:", dict(formatos_pressao))
print("Outros formatos encontrados:", sorted(set(outros_formatos)))

Quantidade por formato: {'dois números': 454, 'interrogação': 17, 'outros': 27, 'vazio': 22}
Outros formatos encontrados: ['103x73', '104x74', '110x84', '112x91', '115x79', '116x68', '116x88', '119x86', '119x89', '120x76', '122x85', '123x89', '125x80', '125x82', '126x87', '127x76', '127x79', '132x78', '135x86', '136x80', '136x90', '140x94', '141x92', '150x88', '159x98', '90x80', '91x70']


#### Padronização da pressão arterial

Padronizamos o separador das medidas para / e removemos espaços.
O símbolo ? é tratado como valor ausente, assim como os campos vazios.

Essa interpretação é uma premissa da análise. Os valores originais
continuam disponíveis em amostras e no arquivo de treinamento.

In [30]:
for amostra in amostras_sem_repeticao:
    pressao = amostra["BLOOD_PRESSURE"].strip()

    if pressao == "?":
        pressao = ""
    else:
        pressao = pressao.replace("x", "/").replace(" ", "")

    amostra["BLOOD_PRESSURE"] = pressao

pressoes_padronizadas = [
    amostra["BLOOD_PRESSURE"]
    for amostra in amostras_sem_repeticao
]

print("Pressões ausentes:", pressoes_padronizadas.count(""))
print("Pressões preenchidas:", len(pressoes_padronizadas) - pressoes_padronizadas.count(""))

Pressões ausentes: 39
Pressões preenchidas: 481


#### Análise das medidas de pressão arterial

Separamos os dois números das pressões preenchidas para observar
seus valores mínimos e máximos.

Essa análise não altera a coluna BLOOD_PRESSURE nem define faixas
de pressão.

In [31]:
pressoes_sistolicas = []
pressoes_diastolicas = []

for pressao in pressoes_padronizadas:
    if pressao != "":
        partes = pressao.split("/")

        sistolica = int(partes[0])
        diastolica = int(partes[1])

        pressoes_sistolicas.append(sistolica)
        pressoes_diastolicas.append(diastolica)

print("Medidas analisadas:", len(pressoes_sistolicas))

print("Menor sistólica:", min(pressoes_sistolicas))
print("Maior sistólica:", max(pressoes_sistolicas))

print("Menor diastólica:", min(pressoes_diastolicas))
print("Maior diastólica:", max(pressoes_diastolicas))

Medidas analisadas: 481
Menor sistólica: 90
Maior sistólica: 175
Menor diastólica: 53
Maior diastólica: 107


#### Separação das medidas de pressão

Criamos as colunas SYSTOLIC e DIASTOLIC a partir de BLOOD_PRESSURE.
Quando a pressão está ausente, as duas novas colunas permanecem vazias.

A coluna original padronizada é mantida para consulta.

In [32]:
for amostra in amostras_sem_repeticao:
    pressao = amostra["BLOOD_PRESSURE"]

    if pressao == "":
        amostra["SYSTOLIC"] = ""
        amostra["DIASTOLIC"] = ""
    else:
        partes = pressao.split("/")
        amostra["SYSTOLIC"] = int(partes[0])
        amostra["DIASTOLIC"] = int(partes[1])

print("Amostras com as novas colunas:", len(amostras_sem_repeticao))

Amostras com as novas colunas: 520


#### Conferência das colunas de pressão

Conferimos a quantidade de valores ausentes nas novas colunas e
comparamos alguns exemplos com BLOOD_PRESSURE.

In [33]:
sistolicas_vazias = sum(
    1 for amostra in amostras_sem_repeticao
    if amostra["SYSTOLIC"] == ""
)

diastolicas_vazias = sum(
    1 for amostra in amostras_sem_repeticao
    if amostra["DIASTOLIC"] == ""
)

print("Sistólicas vazias:", sistolicas_vazias)
print("Diastólicas vazias:", diastolicas_vazias)

print("\nExemplos:")

for amostra in amostras_sem_repeticao[:5]:
    print(
        amostra["BLOOD_PRESSURE"],
        "→", amostra["SYSTOLIC"],
        "e", amostra["DIASTOLIC"]
    )

Sistólicas vazias: 39
Diastólicas vazias: 39

Exemplos:
113/87 → 113 e 87
114/75 → 114 e 75
 →  e 
95/63 → 95 e 63
115/83 → 115 e 83


#### Resumo da pressão arterial

Na base sem repetição, encontramos 481 pressões preenchidas e
39 ausentes, incluindo os registros marcados com ?.

Padronizamos as pressões preenchidas com o separador / e criamos
as colunas numéricas SYSTOLIC e DIASTOLIC. As duas permanecem
vazias quando a pressão não está disponível.

Não encontramos registros com a sistólica menor ou igual à
diastólica. A coluna BLOOD_PRESSURE foi mantida para consulta.

### Verificação do tipo sanguíneo

Contamos os valores registrados em BLOOD_TYPE na base sem repetição.
Essa análise permite identificar categorias, grafias diferentes e
valores ausentes antes da padronização.

In [34]:
tipos_sanguineos = [
    amostra["BLOOD_TYPE"].strip()
    for amostra in amostras_sem_repeticao
]

contagem_tipos = Counter(tipos_sanguineos)

for tipo, quantidade in contagem_tipos.items():
    print(f"{tipo!r}: {quantidade}")

'O+': 189
'B+': 47
'O-': 48
'A+': 164
'': 20
'A-': 26
'AB-': 3
'AB+': 9
'B-': 14


#### Separação do tipo sanguíneo

Criamos as colunas BLOOD_GROUP e RH_FACTOR a partir de BLOOD_TYPE.
Quando o tipo sanguíneo está ausente, as duas novas colunas também
permanecem vazias.

A coluna BLOOD_TYPE é mantida para consulta.

In [35]:
for amostra in amostras_sem_repeticao:
    tipo = amostra["BLOOD_TYPE"].strip()

    if tipo == "":
        amostra["BLOOD_GROUP"] = ""
        amostra["RH_FACTOR"] = ""
    else:
        amostra["BLOOD_GROUP"] = tipo[:-1]
        amostra["RH_FACTOR"] = tipo[-1]

print("Amostras processadas:", len(amostras_sem_repeticao))

Amostras processadas: 520


#### Conferência do grupo sanguíneo e do fator Rh

Contamos os valores de BLOOD_GROUP e RH_FACTOR para conferir a
separação e a quantidade de campos vazios.

In [36]:
contagem_grupos = Counter(
    amostra["BLOOD_GROUP"]
    for amostra in amostras_sem_repeticao
)

contagem_fatores = Counter(
    amostra["RH_FACTOR"]
    for amostra in amostras_sem_repeticao
)

print("Grupos sanguíneos:", dict(contagem_grupos))
print("Fatores Rh:", dict(contagem_fatores))

Grupos sanguíneos: {'O': 237, 'B': 61, 'A': 190, '': 20, 'AB': 12}
Fatores Rh: {'+': 409, '-': 91, '': 20}


#### Resumo do tipo sanguíneo

A coluna BLOOD_TYPE apresenta os oito tipos sanguíneos e 20 valores
ausentes. Criamos BLOOD_GROUP e RH_FACTOR para separar o grupo do
fator Rh, mantendo a coluna original para consulta.

As duas novas colunas também possuem 20 campos vazios. A escolha de
qual representação usar será avaliada nos experimentos.

### Verificação de tabagismo

Contamos os valores registrados em SMOKER para identificar as
categorias, possíveis diferenças de escrita e valores ausentes.

In [37]:
valores_smoker = [
    amostra["SMOKER"].strip()
    for amostra in amostras_sem_repeticao
]

contagem_smoker = Counter(valores_smoker)

for valor, quantidade in contagem_smoker.items():
    print(f"{valor!r}: {quantidade}")

'never': 173
'former': 80
'': 170
'current': 57
'NEVER': 16
'Former': 3
'yes': 4
'ex-smoker': 6
'Never': 7
'Current': 4


#### Padronização de tabagismo

Padronizamos SMOKER em nunca_fumou, ex_fumante e fumante_atual.
Adotamos como premissa que o valor yes indica fumante atual.

Os campos vazios permanecem sem informação.

In [38]:
mapa_smoker = {
    "never": "nunca_fumou",
    "former": "ex_fumante",
    "ex-smoker": "ex_fumante",
    "current": "fumante_atual",
    "yes": "fumante_atual",
    "": ""
}

for amostra in amostras_sem_repeticao:
    valor = amostra["SMOKER"].strip().lower()
    amostra["SMOKER"] = mapa_smoker[valor]

contagem_smoker_padronizado = Counter(
    amostra["SMOKER"]
    for amostra in amostras_sem_repeticao
)

print(dict(contagem_smoker_padronizado))

{'nunca_fumou': 196, 'ex_fumante': 89, '': 170, 'fumante_atual': 65}


#### Resumo de tabagismo

Padronizamos SMOKER em português: nunca_fumou, ex_fumante e
fumante_atual. Adotamos a premissa de que yes indica fumante atual.

Após a padronização, temos 196 registros de nunca fumou, 89 de
ex-fumante, 65 de fumante atual e 170 sem informação.

### Verificação de atividade física

Contamos os valores registrados em PHYSICAL_ACTIVITY para identificar
as categorias, diferenças de escrita e campos vazios.

In [39]:
valores_atividade = [
    amostra["PHYSICAL_ACTIVITY"].strip()
    for amostra in amostras_sem_repeticao
]

contagem_atividade = Counter(valores_atividade)

for valor, quantidade in contagem_atividade.items():
    print(f"{valor!r}: {quantidade}")

'light': 6
'Sedentary': 157
'Moderate': 101
'': 26
'Light': 144
'Intense': 38
'moderate': 4
'LIGHT': 10
'intense': 3
'MODERATE': 3
'INTENSE': 4
'sedentary': 6
'Moderat': 1
'Sedentry': 4
'SEDENTARY': 6
'Ligth': 5
'Intensive': 2


#### Padronização da atividade física

Padronizamos PHYSICAL_ACTIVITY em sedentario, leve, moderada e intensa.
Também tratamos grafias que parecem erros de digitação e interpretamos
Intensive como atividade intensa.

Os campos vazios permanecem sem informação.

In [40]:
mapa_atividade = {
    "sedentary": "sedentario",
    "sedentry": "sedentario",
    "light": "leve",
    "ligth": "leve",
    "moderate": "moderada",
    "moderat": "moderada",
    "intense": "intensa",
    "intensive": "intensa",
    "": ""
}

for amostra in amostras_sem_repeticao:
    valor = amostra["PHYSICAL_ACTIVITY"].strip().lower()
    amostra["PHYSICAL_ACTIVITY"] = mapa_atividade[valor]

contagem_atividade_padronizada = Counter(
    amostra["PHYSICAL_ACTIVITY"]
    for amostra in amostras_sem_repeticao
)

print(dict(contagem_atividade_padronizada))

{'leve': 165, 'sedentario': 173, 'moderada': 109, '': 26, 'intensa': 47}


#### Resumo da atividade física

Padronizamos PHYSICAL_ACTIVITY em sedentario, leve, moderada e intensa.
As grafias Sedentry, Ligth e Moderat foram interpretadas como erros de
digitação; Intensive foi interpretado como atividade intensa.

A base possui 173 registros sedentários, 165 de atividade leve,
109 moderada, 47 intensa e 26 sem informação.

### Verificação do histórico familiar

Contamos os valores registrados em FAMILY_HISTORY para identificar
as categorias, diferenças de escrita e campos vazios antes da
padronização.

In [41]:
valores_historico = [
    amostra["FAMILY_HISTORY"].strip()
    for amostra in amostras_sem_repeticao
]

contagem_historico = Counter(valores_historico)

for valor, quantidade in contagem_historico.items():
    print(f"{valor!r}: {quantidade}")

'No': 250
'YES': 11
'Não': 11
'Yes': 110
'1': 8
'N': 11
'': 16
'Y': 7
'Unknown': 32
'no': 13
'NO': 15
'0': 13
'yes': 15
'Sim': 8


#### Padronização do histórico familiar

Padronizamos FAMILY_HISTORY em sim, nao e desconhecido.

Adotamos como premissa que 1 significa sim e 0 significa não.
Os valores Unknown e os campos vazios são agrupados como desconhecido.

In [42]:
mapa_historico = {
    "yes": "sim",
    "y": "sim",
    "sim": "sim",
    "1": "sim",
    "no": "nao",
    "n": "nao",
    "não": "nao",
    "nao": "nao",
    "0": "nao",
    "unknown": "desconhecido",
    "desconhecido": "desconhecido",
    "": "desconhecido"
}

for amostra in amostras_sem_repeticao:
    valor = amostra["FAMILY_HISTORY"].strip().lower()
    amostra["FAMILY_HISTORY"] = mapa_historico[valor]

contagem_historico_padronizado = Counter(
    amostra["FAMILY_HISTORY"]
    for amostra in amostras_sem_repeticao
)

print(dict(contagem_historico_padronizado))

{'nao': 313, 'sim': 159, 'desconhecido': 48}


#### Resumo do histórico familiar

Padronizamos FAMILY_HISTORY em sim, nao e desconhecido. Os valores 1
e 0 foram interpretados como sim e não, respectivamente.

Os 32 registros Unknown e os 16 campos vazios foram agrupados como
desconhecido. Após a padronização, temos 159 sim, 313 nao e
48 desconhecido.

### Verificação da glicose

Verificamos valores vazios, formatos numéricos e valores que não
puderem ser convertidos na coluna GLUCOSE.

Também observamos os valores mínimo e máximo para orientar a análise.
A unidade ainda precisa ser confirmada.

In [43]:
glicoses_numericas = []
glicoses_nao_convertidas = []
glicoses_vazias = 0

for amostra in amostras_sem_repeticao:
    glicose = amostra["GLUCOSE"].strip()

    if glicose == "":
        glicoses_vazias += 1
    else:
        try:
            glicose_numero = float(glicose.replace(",", "."))
            glicoses_numericas.append(glicose_numero)
        except ValueError:
            glicoses_nao_convertidas.append(glicose)

print("Campos vazios:", glicoses_vazias)
print("Valores numéricos:", len(glicoses_numericas))
print("Valores não convertidos:", sorted(set(glicoses_nao_convertidas)))

if glicoses_numericas:
    print("Menor valor:", min(glicoses_numericas))
    print("Maior valor:", max(glicoses_numericas))

Campos vazios: 59
Valores numéricos: 461
Valores não convertidos: []
Menor valor: 3.1
Maior valor: 500.0


#### Distribuição dos valores de glicose

Calculamos a mediana e os quartis dos valores numéricos de GLUCOSE.
Também observamos os menores e os maiores valores para investigar
possíveis diferenças de escala ou inconsistências.

A análise não altera a coluna nem confirma sua unidade.

In [44]:
q1_glicose, mediana_glicose, q3_glicose = quantiles(
    glicoses_numericas,
    n=4,
    method="inclusive"
)

glicoses_ordenadas = sorted(glicoses_numericas)

print("Primeiro quartil:", q1_glicose)
print("Mediana:", mediana_glicose)
print("Terceiro quartil:", q3_glicose)

print("\nDez menores valores:", glicoses_ordenadas[:10])
print("Dez maiores valores:", glicoses_ordenadas[-10:])

Primeiro quartil: 87.0
Mediana: 104.0
Terceiro quartil: 133.0

Dez menores valores: [3.1, 4.1, 4.3, 4.5, 4.5, 4.8, 4.8, 4.8, 5.0, 5.1]
Dez maiores valores: [297.0, 301.0, 303.0, 315.0, 324.0, 331.0, 364.0, 377.0, 385.0, 500.0]


#### Investigação das escalas de glicose

Contamos os valores de GLUCOSE em quatro intervalos para observar
a distribuição dos números pequenos, intermediários e maiores.

Os limites são usados apenas nesta investigação e não determinam
a unidade dos registros.

In [45]:
glicose_abaixo_20 = [
    glicose for glicose in glicoses_numericas
    if glicose < 20
]

glicose_entre_20_60 = [
    glicose for glicose in glicoses_numericas
    if 20 <= glicose < 60
]

glicose_entre_60_300 = [
    glicose for glicose in glicoses_numericas
    if 60 <= glicose < 300
]

glicose_a_partir_300 = [
    glicose for glicose in glicoses_numericas
    if glicose >= 300
]

print("Abaixo de 20:", len(glicose_abaixo_20))
print("De 20 a menos de 60:", len(glicose_entre_20_60))
print("De 60 a menos de 300:", len(glicose_entre_60_300))
print("A partir de 300:", len(glicose_a_partir_300))

print("\nValores abaixo de 20:", sorted(glicose_abaixo_20))
print("Valores entre 20 e menos de 60:", sorted(glicose_entre_20_60))
print("Valores a partir de 300:", sorted(glicose_a_partir_300))

Abaixo de 20: 43
De 20 a menos de 60: 5
De 60 a menos de 300: 404
A partir de 300: 9

Valores abaixo de 20: [3.1, 4.1, 4.3, 4.5, 4.5, 4.8, 4.8, 4.8, 5.0, 5.1, 5.1, 5.1, 5.1, 5.1, 5.2, 5.2, 5.3, 5.3, 5.3, 5.3, 5.4, 5.5, 5.5, 5.5, 5.6, 5.7, 5.8, 5.8, 5.9, 6.1, 6.2, 6.8, 7.0, 7.2, 7.4, 7.5, 7.5, 8.1, 8.5, 10.0, 10.2, 11.6, 13.3]
Valores entre 20 e menos de 60: [21.3, 47.0, 56.0, 58.0, 59.0]
Valores a partir de 300: [301.0, 303.0, 315.0, 324.0, 331.0, 364.0, 377.0, 385.0, 500.0]


#### Premissa sobre a unidade de glicose

Adotamos provisoriamente a premissa de que todos os valores de
GLUCOSE estão em mg/dL. Nenhuma conversão de unidade foi aplicada.

Os valores muito baixos e altos serão investigados. A unidade não
foi confirmada pela documentação disponível, e essa limitação será
considerada na análise dos resultados.

#### Investigação de valores extremos de glicose

Usamos o intervalo interquartil para identificar valores de GLUCOSE
que se afastam da distribuição observada.

Os valores identificados serão investigados. Estar fora dos limites
estatísticos não significa, por si só, que um valor esteja errado.

In [46]:
intervalo_glicose = q3_glicose - q1_glicose

limite_inferior_glicose = q1_glicose - 1.5 * intervalo_glicose
limite_superior_glicose = q3_glicose + 1.5 * intervalo_glicose

glicoses_fora_limites = [
    glicose for glicose in glicoses_numericas
    if glicose < limite_inferior_glicose
    or glicose > limite_superior_glicose
]

print("Intervalo interquartil:", intervalo_glicose)
print("Limite inferior:", limite_inferior_glicose)
print("Limite superior:", limite_superior_glicose)
print("Quantidade fora dos limites:", len(glicoses_fora_limites))
print("Valores encontrados:", sorted(glicoses_fora_limites))

Intervalo interquartil: 46.0
Limite inferior: 18.0
Limite superior: 202.0
Quantidade fora dos limites: 82
Valores encontrados: [3.1, 4.1, 4.3, 4.5, 4.5, 4.8, 4.8, 4.8, 5.0, 5.1, 5.1, 5.1, 5.1, 5.1, 5.2, 5.2, 5.3, 5.3, 5.3, 5.3, 5.4, 5.5, 5.5, 5.5, 5.6, 5.7, 5.8, 5.8, 5.9, 6.1, 6.2, 6.8, 7.0, 7.2, 7.4, 7.5, 7.5, 8.1, 8.5, 10.0, 10.2, 11.6, 13.3, 204.0, 205.0, 208.0, 208.0, 208.0, 209.0, 215.0, 215.0, 221.0, 224.0, 226.0, 230.0, 232.0, 233.0, 238.0, 241.0, 243.0, 246.0, 247.0, 248.0, 257.0, 262.0, 262.0, 266.0, 267.0, 272.0, 291.0, 293.0, 296.0, 297.0, 301.0, 303.0, 315.0, 324.0, 331.0, 364.0, 377.0, 385.0, 500.0]


#### Padronização numérica da glicose

Convertamos os valores preenchidos de GLUCOSE para números decimais.
Os campos vazios permanecem sem valor.

Mantemos os valores extremos e a premissa provisória de que todos os
registros estão em mg/dL. Nenhuma conversão de unidade é aplicada.

In [47]:
for amostra in amostras_sem_repeticao:
    glicose_texto = str(amostra["GLUCOSE"]).strip().replace(",", ".")

    if glicose_texto != "":
        amostra["GLUCOSE"] = float(glicose_texto)

glicoses_padronizadas = [
    amostra["GLUCOSE"]
    for amostra in amostras_sem_repeticao
]

print("Valores preenchidos:", len(glicoses_padronizadas) - glicoses_padronizadas.count(""))
print("Campos vazios:", glicoses_padronizadas.count(""))

Valores preenchidos: 461
Campos vazios: 59


#### Resumo da glicose

A coluna GLUCOSE possui 461 valores preenchidos e 59 campos vazios.
Adotamos provisoriamente a premissa de que todos os valores estão
em mg/dL, sem conversão de unidade.

Pelo critério do intervalo interquartil, 82 valores ficaram fora
dos limites estatísticos: 43 abaixo de 18 e 39 acima de 202.
Todos foram mantidos para avaliação posterior.

### Verificação da hemoglobina

Verificamos os campos vazios, os valores numéricos e os valores que
não puderem ser convertidos na coluna HEMOGLOBIN.

Também observamos os valores mínimo e máximo. O tipo de medida e sua
unidade ainda precisam ser investigados.

In [48]:
hemoglobinas_numericas = []
hemoglobinas_nao_convertidas = []
hemoglobinas_vazias = 0

for amostra in amostras_sem_repeticao:
    hemoglobina = amostra["HEMOGLOBIN"].strip()

    if hemoglobina == "":
        hemoglobinas_vazias += 1
    else:
        try:
            valor = float(hemoglobina.replace(",", "."))
            hemoglobinas_numericas.append(valor)
        except ValueError:
            hemoglobinas_nao_convertidas.append(hemoglobina)

print("Campos vazios:", hemoglobinas_vazias)
print("Valores numéricos:", len(hemoglobinas_numericas))
print(
    "Valores não convertidos:",
    sorted(set(hemoglobinas_nao_convertidas))
)

if hemoglobinas_numericas:
    print("Menor valor:", min(hemoglobinas_numericas))
    print("Maior valor:", max(hemoglobinas_numericas))

Campos vazios: 122
Valores numéricos: 398
Valores não convertidos: []
Menor valor: 4.7
Maior valor: 116.0


#### Distribuição da hemoglobina

Calculamos a mediana e os quartis dos valores preenchidos de HEMOGLOBIN. Também observamos os dez menores e os dez maiores valores.

Essa análise não altera a coluna. O tipo de medida e sua unidade permanecem sem confirmação.


In [49]:
q1_hemoglobina, mediana_hemoglobina, q3_hemoglobina = quantiles(
    hemoglobinas_numericas,
    n=4,
    method="inclusive"
)

hemoglobinas_ordenadas = sorted(hemoglobinas_numericas)

print("Primeiro quartil:", q1_hemoglobina)
print("Mediana:", mediana_hemoglobina)
print("Terceiro quartil:", q3_hemoglobina)
print("\nDez menores valores:", hemoglobinas_ordenadas[:10])
print("Dez maiores valores:", hemoglobinas_ordenadas[-10:])


Primeiro quartil: 5.5
Mediana: 6.4
Terceiro quartil: 8.7

Dez menores valores: [4.7, 4.8, 4.8, 4.8, 4.8, 4.8, 4.8, 4.8, 4.8, 4.8]
Dez maiores valores: [58.0, 60.0, 63.0, 64.0, 65.0, 66.0, 67.0, 69.0, 88.0, 116.0]


#### Investigação das escalas e dos valores extremos

Contamos os valores em intervalos exploratórios para observar possíveis diferenças de escala. Os limites não identificam unidades nem classificam resultados clínicos. Também usamos o intervalo interquartil para identificar valores que se afastam da distribuição; isso não confirma erros.


In [50]:
faixas_hemoglobina = {
    "abaixo de 20": [],
    "de 20 a menos de 40": [],
    "a partir de 40": []
}

for valor in hemoglobinas_numericas:
    if valor < 20:
        faixas_hemoglobina["abaixo de 20"].append(valor)
    elif valor < 40:
        faixas_hemoglobina["de 20 a menos de 40"].append(valor)
    else:
        faixas_hemoglobina["a partir de 40"].append(valor)

for faixa, valores in faixas_hemoglobina.items():
    print(f"{faixa}: {len(valores)}")
    if valores:
        print(f"  Menor: {min(valores)} | Maior: {max(valores)}")

intervalo_hemoglobina = q3_hemoglobina - q1_hemoglobina
limite_inferior_hemoglobina = q1_hemoglobina - 1.5 * intervalo_hemoglobina
limite_superior_hemoglobina = q3_hemoglobina + 1.5 * intervalo_hemoglobina

fora_limites_hemoglobina = [
    valor for valor in hemoglobinas_numericas
    if valor < limite_inferior_hemoglobina
    or valor > limite_superior_hemoglobina
]

print("Limites estatísticos:", limite_inferior_hemoglobina, "a", limite_superior_hemoglobina)
print("Valores fora dos limites:", len(fora_limites_hemoglobina))


abaixo de 20: 355
  Menor: 4.7 | Maior: 13.1
de 20 a menos de 40: 23
  Menor: 27.0 | Maior: 39.0
a partir de 40: 20
  Menor: 40.0 | Maior: 116.0
Limites estatísticos: 0.7000000000000011 a 13.499999999999998
Valores fora dos limites: 43


#### Padronização numérica da hemoglobina

Convertemos os valores preenchidos de HEMOGLOBIN para números decimais e mantemos os campos vazios. Preservamos os valores extremos e não convertemos unidades: o tipo de medida e as unidades de cada registro não foram confirmados.


In [51]:
for amostra in amostras_sem_repeticao:
    texto = str(amostra["HEMOGLOBIN"]).strip().replace(",", ".")

    if texto != "":
        amostra["HEMOGLOBIN"] = float(texto)

hemoglobinas_padronizadas = [
    amostra["HEMOGLOBIN"]
    for amostra in amostras_sem_repeticao
]

print("Valores preenchidos:", len(hemoglobinas_padronizadas) - hemoglobinas_padronizadas.count(""))
print("Campos vazios:", hemoglobinas_padronizadas.count(""))


Valores preenchidos: 398
Campos vazios: 122


#### Resumo da hemoglobina

Na base sem repetição, HEMOGLOBIN possui 398 valores preenchidos e 122 campos vazios. A mediana é 6,4; os quartis são 5,5 e 8,7.

Encontramos 355 valores abaixo de 20, 23 entre 20 e menos de 40 e 20 a partir de 40. Pelo critério interquartil, 43 valores ficaram fora dos limites aproximados de 0,7 a 13,5.

Mantemos HEMOGLOBIN como uma única coluna numérica, com os valores altos preservados para avaliação nos experimentos. Não aplicamos conversão de unidade. O significado da medida e sua unidade não foram confirmados.


### Verificação de triglicerídeos

Examinamos os formatos, campos vazios e a distribuição de TRIGLYCERIDES na base sem repetição. A unidade e o contexto da medição não estão confirmados.


In [52]:
triglicerideos_numericos = []
triglicerideos_nao_convertidos = []
triglicerideos_vazios = 0

for amostra in amostras_sem_repeticao:
    texto = str(amostra["TRIGLYCERIDES"]).strip()

    if texto == "":
        triglicerideos_vazios += 1
    else:
        try:
            triglicerideos_numericos.append(float(texto.replace(",", ".")))
        except ValueError:
            triglicerideos_nao_convertidos.append(texto)

print("Campos vazios:", triglicerideos_vazios)
print("Valores numéricos:", len(triglicerideos_numericos))
print("Valores não convertidos:", sorted(set(triglicerideos_nao_convertidos)))


Campos vazios: 44
Valores numéricos: 476
Valores não convertidos: []


#### Distribuição e valores extremos de triglicerídeos

Calculamos mediana, quartis e limites pelo intervalo interquartil. Os valores fora dos limites merecem investigação; não são necessariamente erros e serão mantidos.


In [53]:
if len(triglicerideos_numericos) >= 2:
    q1_triglicerideos, mediana_triglicerideos, q3_triglicerideos = quantiles(
        triglicerideos_numericos, n=4, method="inclusive"
    )
    intervalo_triglicerideos = q3_triglicerideos - q1_triglicerideos
    limite_inferior_triglicerideos = q1_triglicerideos - 1.5 * intervalo_triglicerideos
    limite_superior_triglicerideos = q3_triglicerideos + 1.5 * intervalo_triglicerideos
    fora_limites_triglicerideos = [
        valor for valor in triglicerideos_numericos
        if valor < limite_inferior_triglicerideos or valor > limite_superior_triglicerideos
    ]
    ordenados_triglicerideos = sorted(triglicerideos_numericos)

    print("Mínimo:", min(triglicerideos_numericos))
    print("Q1:", q1_triglicerideos, "| Mediana:", mediana_triglicerideos, "| Q3:", q3_triglicerideos)
    print("Máximo:", max(triglicerideos_numericos))
    print("Limites estatísticos:", limite_inferior_triglicerideos, "a", limite_superior_triglicerideos)
    print("Quantidade fora dos limites:", len(fora_limites_triglicerideos))
    print("Dez menores:", ordenados_triglicerideos[:10])
    print("Dez maiores:", ordenados_triglicerideos[-10:])


Mínimo: 21.0
Q1: 106.75 | Mediana: 150.0 | Q3: 203.25
Máximo: 2880.0
Limites estatísticos: -38.0 a 348.0
Quantidade fora dos limites: 17
Dez menores: [21.0, 42.0, 43.0, 44.0, 47.0, 47.0, 47.0, 48.0, 51.0, 52.0]
Dez maiores: [407.0, 566.0, 580.0, 790.0, 800.0, 820.0, 1000.0, 1540.0, 2350.0, 2880.0]


#### Padronização numérica de triglicerídeos

Convertemos os valores preenchidos de TRIGLYCERIDES para números decimais, preservando os campos vazios. Não alteramos valores extremos nem convertemos unidades.


In [54]:
for amostra in amostras_sem_repeticao:
    texto = str(amostra["TRIGLYCERIDES"]).strip().replace(",", ".")

    if texto != "":
        amostra["TRIGLYCERIDES"] = float(texto)

print("Valores preenchidos:", sum(1 for amostra in amostras_sem_repeticao if amostra["TRIGLYCERIDES"] != ""))
print("Campos vazios:", sum(1 for amostra in amostras_sem_repeticao if amostra["TRIGLYCERIDES"] == ""))


Valores preenchidos: 476
Campos vazios: 44


#### Resumo dos triglicerídeos

A base sem repetição contém 476 valores preenchidos e 44 vazios. Os valores numéricos variam de 21 a 2880. Os extremos permanecem na base; sua unidade e origem não foram confirmadas. A mediana, os quartis e a contagem de possíveis extremos estão registrados acima.


### Verificação de sódio

Examinamos os formatos, campos vazios e a distribuição de SODIUM na base sem repetição. A unidade e o contexto da medição não estão confirmados.


In [55]:
sodio_numericos = []
sodio_nao_convertidos = []
sodio_vazios = 0

for amostra in amostras_sem_repeticao:
    texto = str(amostra["SODIUM"]).strip()

    if texto == "":
        sodio_vazios += 1
    else:
        try:
            sodio_numericos.append(float(texto.replace(",", ".")))
        except ValueError:
            sodio_nao_convertidos.append(texto)

print("Campos vazios:", sodio_vazios)
print("Valores numéricos:", len(sodio_numericos))
print("Valores não convertidos:", sorted(set(sodio_nao_convertidos)))


Campos vazios: 12
Valores numéricos: 508
Valores não convertidos: []


#### Distribuição e valores extremos de sódio

Calculamos mediana, quartis e limites pelo intervalo interquartil. Os valores fora dos limites merecem investigação; não são necessariamente erros e serão mantidos.


In [56]:
if len(sodio_numericos) >= 2:
    q1_sodio, mediana_sodio, q3_sodio = quantiles(
        sodio_numericos, n=4, method="inclusive"
    )
    intervalo_sodio = q3_sodio - q1_sodio
    limite_inferior_sodio = q1_sodio - 1.5 * intervalo_sodio
    limite_superior_sodio = q3_sodio + 1.5 * intervalo_sodio
    fora_limites_sodio = [
        valor for valor in sodio_numericos
        if valor < limite_inferior_sodio or valor > limite_superior_sodio
    ]
    ordenados_sodio = sorted(sodio_numericos)

    print("Mínimo:", min(sodio_numericos))
    print("Q1:", q1_sodio, "| Mediana:", mediana_sodio, "| Q3:", q3_sodio)
    print("Máximo:", max(sodio_numericos))
    print("Limites estatísticos:", limite_inferior_sodio, "a", limite_superior_sodio)
    print("Quantidade fora dos limites:", len(fora_limites_sodio))
    print("Dez menores:", ordenados_sodio[:10])
    print("Dez maiores:", ordenados_sodio[-10:])


Mínimo: 133.0
Q1: 138.0 | Mediana: 140.0 | Q3: 142.0
Máximo: 149.0
Limites estatísticos: 132.0 a 148.0
Quantidade fora dos limites: 1
Dez menores: [133.0, 133.0, 134.0, 134.0, 134.0, 134.0, 135.0, 135.0, 135.0, 135.0]
Dez maiores: [146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 146.0, 147.0, 148.0, 149.0]


#### Padronização numérica de sódio

Convertemos os valores preenchidos de SODIUM para números decimais, preservando os campos vazios. Não alteramos valores extremos nem convertemos unidades.


In [57]:
for amostra in amostras_sem_repeticao:
    texto = str(amostra["SODIUM"]).strip().replace(",", ".")

    if texto != "":
        amostra["SODIUM"] = float(texto)

print("Valores preenchidos:", sum(1 for amostra in amostras_sem_repeticao if amostra["SODIUM"] != ""))
print("Campos vazios:", sum(1 for amostra in amostras_sem_repeticao if amostra["SODIUM"] == ""))


Valores preenchidos: 508
Campos vazios: 12


#### Resumo do sódio

A base sem repetição contém 508 valores preenchidos e 12 vazios. Os valores numéricos variam de 133 a 149. Os extremos permanecem na base; a unidade não foi confirmada. A mediana, os quartis e a contagem de possíveis extremos estão registrados acima.


### Verificação do local de coleta

Contamos os valores de COLLECTION_SITE, inclusive campos vazios. Pelo nome, a coluna pode indicar o local de coleta, mas o significado dos códigos não foi confirmado.


In [58]:
locais = [amostra["COLLECTION_SITE"].strip() for amostra in amostras_sem_repeticao]
contagem_locais = Counter(locais)

print("Campos vazios:", contagem_locais.get("", 0))
print("Códigos distintos preenchidos:", len([local for local in contagem_locais if local != ""]))
print("Quantidade por código:")
for local, quantidade in sorted(contagem_locais.items()):
    print(f"{local!r}: {quantidade}")


Campos vazios: 0
Códigos distintos preenchidos: 52
Quantidade por código:
'SITE-001': 141
'SITE-002': 70
'SITE-003': 43
'SITE-004': 25
'SITE-005': 23
'SITE-006': 20
'SITE-007': 21
'SITE-008': 7
'SITE-009': 14
'SITE-010': 9
'SITE-011': 9
'SITE-012': 8
'SITE-013': 8
'SITE-014': 11
'SITE-015': 6
'SITE-016': 5
'SITE-017': 3
'SITE-018': 5
'SITE-019': 7
'SITE-020': 4
'SITE-021': 4
'SITE-022': 2
'SITE-023': 1
'SITE-024': 8
'SITE-025': 5
'SITE-026': 3
'SITE-027': 1
'SITE-028': 1
'SITE-029': 5
'SITE-030': 3
'SITE-031': 3
'SITE-032': 1
'SITE-035': 1
'SITE-036': 1
'SITE-038': 4
'SITE-039': 1
'SITE-040': 2
'SITE-042': 4
'SITE-043': 1
'SITE-044': 2
'SITE-045': 1
'SITE-047': 5
'SITE-048': 4
'SITE-049': 2
'SITE-050': 2
'SITE-052': 2
'SITE-053': 3
'SITE-054': 3
'SITE-055': 2
'SITE-057': 1
'SITE-058': 2
'SITE-060': 1


#### Resumo do local de coleta

Encontramos 52 códigos distintos preenchidos e nenhum campo vazio. Mantemos os valores como recebidos. Antes de usar a coluna em um modelo, precisamos avaliar seu significado e se ela poderia refletir características da coleta ou da origem das amostras.


### Verificação do equipamento de laboratório

Contamos os valores de LAB_DEVICE para identificar categorias e ausências. O significado dos códigos e as diferenças entre equipamentos ainda não foram documentados.


In [59]:
equipamentos = [amostra["LAB_DEVICE"].strip() for amostra in amostras_sem_repeticao]
contagem_equipamentos = Counter(equipamentos)

for equipamento, quantidade in sorted(contagem_equipamentos.items()):
    print(f"{equipamento!r}: {quantidade}")
print("Campos vazios:", contagem_equipamentos.get("", 0))


'FTIR-A': 500
'FTIR-B': 20
Campos vazios: 0


#### Resumo do equipamento

Encontramos dois códigos: FTIR-A em 500 amostras e FTIR-B em 20, sem campos vazios. Mantemos a coluna como recebida. Nos experimentos, será necessário avaliar cuidadosamente se usá-la melhora a generalização ou faz o modelo depender do equipamento.


### Verificação do protocolo de estudo

Contamos os valores de STUDY_PROTOCOL para conferir se a coluna apresenta variação ou campos ausentes.


In [60]:
protocolos = [amostra["STUDY_PROTOCOL"].strip() for amostra in amostras_sem_repeticao]
contagem_protocolos = Counter(protocolos)

for protocolo, quantidade in sorted(contagem_protocolos.items()):
    print(f"{protocolo!r}: {quantidade}")
print("Campos vazios:", contagem_protocolos.get("", 0))
print("Protocolos distintos preenchidos:", len([protocolo for protocolo in contagem_protocolos if protocolo != ""]))


'T2D-SPEC-v1': 520
Campos vazios: 0
Protocolos distintos preenchidos: 1


#### Resumo do protocolo de estudo

O treinamento sem repetição apresenta somente T2D-SPEC-v1 nas 520 amostras e não possui campos vazios. Como não varia nesse conjunto, a coluna não diferencia as amostras de treinamento. Mantemos o campo na base padronizada e deixamos a decisão de uso para os modelos.


#### Padronização das idades não informadas

Tratamos as idades 0, -1 e 999 como não informadas, representando-as por campo vazio, como os registros que já estavam sem idade. Não preenchemos as idades ausentes nesta etapa.

In [61]:
for amostra in amostras_sem_repeticao:
    idade = str(amostra["AGE"]).strip()

    if idade in {"0", "-1", "999"}:
        amostra["AGE"] = ""

idades_vazias_finais = sum(
    1 for amostra in amostras_sem_repeticao
    if str(amostra["AGE"]).strip() == ""
)

print("Idades não informadas:", idades_vazias_finais)

Idades não informadas: 11


### Distribuição das classes na base sem repetição

Recontamos as classes depois da retirada das amostras repetidas. A coluna class identifica controle (0) e diabetes (1).


In [62]:
classes_finais = Counter(
    amostra["class"].strip()
    for amostra in amostras_sem_repeticao
)

total_final = len(amostras_sem_repeticao)

for classe, quantidade in sorted(classes_finais.items()):
    percentual = quantidade / total_final * 100
    print(f"Classe {classe}: {quantidade} amostras ({percentual:.1f}%)")

print("Total de amostras:", total_final)


Classe 0: 250 amostras (48.1%)
Classe 1: 270 amostras (51.9%)
Total de amostras: 520


### Valores ausentes na base sem repetição

Contamos os campos vazios por coluna após as transformações realizadas. Categorias como desconhecido não são contadas como vazias; elas são apresentadas separadamente na conferência seguinte.


In [63]:
colunas_finais = list(amostras_sem_repeticao[0].keys())
vazios_por_coluna = {}

for coluna in colunas_finais:
    quantidade = sum(
        1 for amostra in amostras_sem_repeticao
        if amostra[coluna] is None or str(amostra[coluna]).strip() == ""
    )
    vazios_por_coluna[coluna] = quantidade
    print(f"{coluna}: {quantidade} campos vazios")


id: 0 campos vazios
PATIENT_CODE: 0 campos vazios
EXAM_DATE: 0 campos vazios
COLLECTION_SITE: 0 campos vazios
LAB_DEVICE: 0 campos vazios
STUDY_PROTOCOL: 0 campos vazios
GENDER: 0 campos vazios
AGE: 11 campos vazios
BIRTH_YEAR: 10 campos vazios
HEIGHT: 34 campos vazios
WEIGHT: 13 campos vazios
BLOOD_PRESSURE: 39 campos vazios
BLOOD_TYPE: 20 campos vazios
SMOKER: 170 campos vazios
PHYSICAL_ACTIVITY: 26 campos vazios
FAMILY_HISTORY: 0 campos vazios
GLUCOSE: 59 campos vazios
HEMOGLOBIN: 122 campos vazios
TRIGLYCERIDES: 44 campos vazios
SODIUM: 12 campos vazios
class: 0 campos vazios
SYSTOLIC: 39 campos vazios
DIASTOLIC: 39 campos vazios
BLOOD_GROUP: 20 campos vazios
RH_FACTOR: 20 campos vazios


### Comparação entre a base original e a base padronizada

A base original é o arquivo de treinamento. A base padronizada é a lista `amostras_sem_repeticao`, construída em memória neste notebook.

| Item ou coluna | Base original | Base padronizada |
| --- | --- | --- |
| Amostras | 540 linhas; 20 pares iguais após padronização, exceto pelo `id`. | 520 linhas; mantemos a primeira amostra de cada paciente. |
| Classes | 259 controle (`0`) e 281 diabetes (`1`). | 250 controle (`0`) e 270 diabetes (`1`). |
| `id` | Identificador de cada amostra. | Mantido o identificador da linha preservada. |
| `PATIENT_CODE` | Código do paciente; 20 códigos apareciam duas vezes. | Mantido, com uma amostra por código. |
| `EXAM_DATE` | Três formatos: `AAAA-MM-DD` (ex.: `2021-07-13`), `DD/MM/AAAA` (ex.: `13/07/2021`) e `Mês DD AAAA` (ex.: `Jul 13 2021`). | Datas no formato `AAAA-MM-DD`. |
| `COLLECTION_SITE` | Códigos associados à coleta. | Mantido como recebido; 52 códigos distintos. |
| `LAB_DEVICE` | Dois códigos de equipamento. | Mantido como recebido: `FTIR-A` e `FTIR-B`. |
| `STUDY_PROTOCOL` | Código de protocolo. | Mantido como recebido; um único valor nas 520 amostras. |
| `GENDER` | Grafias como `MALE`, `female`, `F` e campos vazios. | `M`, `F` ou `N` para não informado. |
| `AGE` | Idades como texto, incluindo campos vazios, `0`, `-1` e `999`. | Valores `0`, `-1` e `999` representados como campos vazios, junto às idades não informadas. Os demais valores permanecem como texto. |
| `BIRTH_YEAR` | Ano de nascimento, com campos vazios. | Mantido como recebido. |
| `HEIGHT` | Valores como `170`, `1.70` e `1,70`. | Valores preenchidos convertidos para números em centímetros; vazios preservados. |
| `WEIGHT` | Pesos como texto, campos vazios e valores extremos. | Valores preenchidos convertidos para números decimais; 13 vazios e extremos preservados. Assumimos provisoriamente kg, sem conversão de unidade. |
| `BLOOD_PRESSURE` | Formatos como `114 / 75`, `120x76`, `?` e vazio. | Texto padronizado, como `114/75`; `?` tratado como vazio. |
| `SYSTOLIC` e `DIASTOLIC` | Não existiam. | Criadas a partir de `BLOOD_PRESSURE`, com números inteiros ou vazio. |
| `BLOOD_TYPE` | Tipos como `A+` e `O-`, além de campos vazios. | Mantido para consulta; os valores preenchidos já seguiam o mesmo padrão. |
| `BLOOD_GROUP` e `RH_FACTOR` | Não existiam. | Criadas a partir de `BLOOD_TYPE`; vazias quando o tipo não é informado. |
| `SMOKER` | Grafias como `never`, `Former`, `current`, `yes` e vazio. | `nunca_fumou`, `ex_fumante`, `fumante_atual` ou vazio. Assumimos que `yes` significa fumante atual. |
| `PHYSICAL_ACTIVITY` | Grafias como `Light`, `Ligth`, `Moderat` e `Intensive`. | `sedentario`, `leve`, `moderada`, `intensa` ou vazio. |
| `FAMILY_HISTORY` | Respostas como `Yes`, `Não`, `1`, `0`, `Unknown` e vazio. | `sim`, `nao` ou `desconhecido`. Assumimos `1 = sim` e `0 = não`; `Unknown` e vazio foram agrupados. |
| `GLUCOSE` | Valores como texto e campos vazios. | Preenchidos convertidos para números decimais; extremos e vazios mantidos. Assumimos provisoriamente mg/dL para todos, sem conversão de unidade. |
| `HEMOGLOBIN` | Valores como texto e campos vazios. | Preenchidos convertidos para números decimais; extremos e vazios mantidos, sem conversão de unidade. |
| `TRIGLYCERIDES` | Valores como texto e campos vazios. | Preenchidos convertidos para números decimais; extremos e vazios mantidos, sem conversão de unidade. |
| `SODIUM` | Valores como texto e campos vazios. | Preenchidos convertidos para números decimais; extremos e vazios mantidos, sem conversão de unidade. |
| `class` | Rótulo `0` ou `1`. | Mantido para cada amostra preservada. |

O tratamento dos campos numéricos ausentes, o escalonamento e as faixas etárias serão decididos durante a construção dos modelos. O notebook não altera o arquivo original e ainda não salva um CSV padronizado.

### Exportação da base de treinamento padronizada

Salvamos as 520 amostras padronizadas em um novo arquivo CSV dentro da pasta dados. O arquivo original permanece preservado.

Campos ausentes permanecem vazios. As regras que dependem de treinamento, como preenchimento por mediana e escalonamento, serão tratadas nos modelos.


In [ ]:
caminho_saida = Path("treinamento_padronizado.csv")
colunas_saida = list(amostras_sem_repeticao[0].keys())

with caminho_saida.open("w", encoding="utf-8-sig", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=colunas_saida)
    escritor.writeheader()
    escritor.writerows(amostras_sem_repeticao)

print("Arquivo gerado:", caminho_saida.resolve())
print("Amostras:", len(amostras_sem_repeticao))
print("Colunas:", len(colunas_saida))


Arquivo gerado: C:\Users\gusta\TRAB_MDADOS\dados\treinamento_padronizado.csv
Amostras: 520
Colunas: 25
